# Neuro-Symbolic Causal Inference Using Transformer-Based Relation Extraction and Bayesian Reasoning
**Deep Learning extension of the AML project** — DistilBERT relation extraction → evidence graph → Bayesian network.

> **How to read the outputs in this notebook.** The notebook is executable top to bottom. It first tries to download the pretrained
> `distilbert-base-uncased`. If that is impossible (no Hugging Face access) it falls back to an *offline smoke mode*: a small **randomly-initialised**
> model with the DistilBERT architecture. Smoke mode only proves the pipeline runs; its scores are **not** the result of transfer learning.
> The banner printed in the setup cell tells you which mode produced the numbers you are looking at. Re-run on your MacBook (MPS) for the real experiment.

## 1. Project introduction
The AML project extracts cause-effect claims from diabetes abstracts (Europe PMC) with spaCy rules, a keyword heuristic and prompted LLMs, then builds an evidence graph.
An audit found that `distilbert_extractor.py` was a keyword heuristic, not DistilBERT, and that the 250 diabetes labels are rule-generated silver labels (only 4 causal, none in dev/test).
This DL extension adds a **genuinely fine-tunable DistilBERT** extractor, trains it on a **human-annotated gold dataset**, compares it with the AML extractors, plugs it into the graph builder and adds a **pgmpy Bayesian network** layer.

## 2. Problem statement
Given a sentence and two marked entity mentions, decide whether the sentence asserts that one **causes** the other, and in which direction:
`NONE`, `CAUSE_E1_E2` (E1 causes E2) or `CAUSE_E2_E1` (E2 causes E1).

## 3. Literature and methodology
* **Transformers / attention** — Vaswani et al., 2017. **BERT** — Devlin et al., 2019. **DistilBERT** — Sanh et al., 2019 (40% smaller, ~60% faster, retains ~97% of BERT's GLUE performance as reported by the authors).
* **Entity-aware relation classification** — Wu & He (R-BERT), 2019 and Baldini Soares et al. (Matching the Blanks), 2019: marking entities and pooling hidden states at entity positions. We use entity-start-marker pooling.
* **Gold data** — SemEval-2010 Task 8 (Hendrickx et al., 2010).
* **Causal reasoning** — Pearl, 2009; Bayesian networks — Koller & Friedman, 2009; **pgmpy** — Ankan & Panda, 2015.

Method: `[E1] … [/E1]`/`[E2] … [/E2]` markers → DistilBERT → concat(h[CLS], h[E1], h[E2]) → dropout → linear (3 classes). Weighted cross-entropy, AdamW (lr 2e-5, wd 0.01), linear warm-up/decay, early stopping on dev macro-F1.

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from src.extractors.distilbert.model import pick_device
DEVICE = pick_device()

# ---- detect whether pretrained weights are obtainable ----
PRETRAINED = False
if not os.environ.get("FORCE_SMOKE"):
    try:
        from transformers import AutoTokenizer
        AutoTokenizer.from_pretrained("distilbert-base-uncased"); PRETRAINED = True
    except Exception as e:
        print("Could not load pretrained distilbert-base-uncased:", type(e).__name__)
TAG = "distilbert_base_uncased" if PRETRAINED else "smoke_random_init"
CKPT = ROOT / "models" / TAG
print("=" * 78)
print("MODE:", "PRETRAINED distilbert-base-uncased (real transfer learning)" if PRETRAINED else
      "OFFLINE SMOKE MODE - random-init small DistilBERT-architecture model. NOT transfer learning.")
print("device:", DEVICE, "| torch", torch.__version__)
print("=" * 78)

ModuleNotFoundError: No module named 'torch'

## 4. Dataset description
**SemEval-2010 Task 8** (human-annotated). 9 directed relation types + `Other`; each sentence has two nominals marked `<e1>`, `<e2>`.
`Cause-Effect(e1,e2)` / `Cause-Effect(e2,e1)` are genuine causal relations with direction. Mapping used here (documented, deterministic):

| original label | project label |
|---|---|
| Cause-Effect(e1,e2) | CAUSE_E1_E2 |
| Cause-Effect(e2,e1) | CAUSE_E2_E1 |
| all other relations incl. Other | NONE |

SemEval has **no** association-only class, so the model never learns it. Corr2Cause (synthetic causal-graph inference from correlation statements) and BioRED (biomedical relations without an explicit causal label) were examined and rejected for this objective — see `reports/dataset_audit.md`.
The diabetes silver labels are kept **only** for the AML/DL comparison, always marked as silver.

In [ ]:
from src.preprocessing.semeval import prepare, load_split, LABELS
data_dir = ROOT / "data/processed"
rep = prepare(ROOT / "data/raw", data_dir, dev_fraction=0.1, seed=42)   # downloads if needed; deterministic
train, dev, test = (load_split(data_dir, s) for s in ("train", "dev", "test"))
print({k: len(v) for k, v in [("train", train), ("dev", dev), ("test", test)]})
print("quality:", {k: rep[k] for k in ("empty_text", "missing_markers", "overlap_train_dev", "overlap_train_test", "test_dropped_for_leakage")})

## 5–6. Dataset exploration and class distribution

In [ ]:
df = pd.DataFrame([{**r, "split": s} for s, rows in [("train", train), ("dev", dev), ("test", test)] for r in rows])
df["n_words"] = df["plain_text"].str.split().str.len()
dist = df.groupby(["split", "label"]).size().unstack(fill_value=0)[LABELS]
display(dist)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
dist.T.plot.bar(ax=ax[0], title="Class distribution per split (heavily imbalanced)"); ax[0].set_ylabel("sentences")
df["n_words"].hist(bins=40, ax=ax[1]); ax[1].set_title("Sentence length (words)"); plt.tight_layout(); plt.show()
print("original SemEval relation types in train+dev:"); display(df[df.split != "test"].original_label.str.replace(r"\(.*\)", "", regex=True).value_counts().to_frame("count").T)

## 7–8. Preprocessing and entity marking
Raw `<e1>…</e1>` tags become the special tokens `[E1] … [/E1]`, `[E2] … [/E2]`. For the diabetes candidate pairs the same markers are inserted from character offsets (`insert_markers_by_offsets`), which rejects overlapping entity spans instead of mis-marking them.

In [ ]:
from src.extractors.distilbert.markers import insert_markers_by_offsets
print(train[0]["text"], "->", train[0]["label"])
s = "Smoking increases the risk of lung cancer"
a, b = s.index("Smoking"), s.index("lung cancer")
print(insert_markers_by_offsets(s, a, a + len("Smoking"), b, b + len("lung cancer")))
print(insert_markers_by_offsets(s, b, b + len("lung cancer"), a, a + len("Smoking")), " (E1/E2 stay tied to the entity, not to position)")

## 9. Tokenisation
The four markers are added as *special tokens* (never split into word pieces) and the embedding matrix is resized. Padding is dynamic per batch; positions of `[E1]`/`[E2]` are located for entity pooling.

In [ ]:
from src.extractors.distilbert.model import build_pretrained, build_offline_smoke, load_checkpoint
from src.extractors.distilbert.dataset import RelationDataset, make_collate, marker_ids
if PRETRAINED:
    tok, _m = build_pretrained("distilbert-base-uncased", len(LABELS))
else:
    tok, _m = build_offline_smoke([r["text"] for r in train], len(LABELS), ROOT / "models/_nb_tok")
ex = train[0]["text"]
print(tok.tokenize(ex)); print("marker ids:", marker_ids(tok)); print("vocab size:", len(tok))
lens = [len(x) for x in tok([r["text"] for r in train], truncation=False)["input_ids"]]
print("token length  mean %.1f | 95th pct %d | max %d  (max_length=128 truncates %.2f%%)" % (np.mean(lens), np.percentile(lens, 95), max(lens), 100*np.mean(np.array(lens) > 128)))

## 10. DistilBERT architecture
DistilBERT (Sanh et al., 2019) keeps BERT's Transformer encoder but with **6 layers** instead of 12 (hidden 768, 12 heads, ~66M parameters), trained by knowledge distillation. Our head concatenates `[CLS]`, `[E1]` and `[E2]` hidden states (3×768) → dropout → linear(3).

In [ ]:
model = _m
n_enc = sum(p.numel() for p in model.encoder.parameters()); n_head = sum(p.numel() for p in model.classifier.parameters())
cfg = model.encoder.config
print(f"encoder layers={cfg.n_layers} hidden={cfg.dim} heads={cfg.n_heads} | encoder params={n_enc:,} | head params={n_head:,}")
print(model.classifier)

## 11. Model training
Hyper-parameters: lr 2e-5, batch 16, 4 epochs, max length 128, AdamW (wd 0.01), warm-up 10%, class-weighted cross-entropy, early stopping patience 2 (pretrained mode). Smoke mode uses a higher learning rate because the model is tiny and trained from scratch.

In [ ]:
from dl.training import train as T
if PRETRAINED:
    args = ["--model", "distilbert-base-uncased", "--epochs", "4", "--batch-size", "16", "--lr", "2e-5"]
else:
    args = ["--offline-smoke", "--epochs", "15", "--patience", "4", "--lr", "5e-4", "--batch-size", "32"]
t0 = time.time(); T.main(args); print("wall-clock training seconds: %.1f" % (time.time() - t0))

## 12. Training and validation loss

In [ ]:
summ = json.load(open(ROOT / f"dl/results/training_summary_{TAG}.json"))
h = pd.DataFrame(summ["history"]); display(h.round(4))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
ax[0].plot(h.epoch, h.train_loss, "o-", label="train"); ax[0].plot(h.epoch, h.val_loss, "s-", label="validation"); ax[0].set_title("Loss"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot(h.epoch, h.val_macro_f1, "o-g"); ax[1].set_title("Validation macro-F1"); ax[1].set_xlabel("epoch"); plt.tight_layout(); plt.show()
print("class weights used:", np.round(summ["class_weights"], 3), "| pretrained:", summ["pretrained"], "| device:", summ["device"])

## 13–15. Evaluation on the SemEval GOLD test split
The checkpoint with the best validation macro-F1 is reloaded and evaluated once on the held-out official test sentences.

In [ ]:
from src.extractors.distilbert.extractor import FineTunedDistilBERTExtractor
from dl.evaluation.common import metrics, semeval_to_pair
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
det = FineTunedDistilBERTExtractor(CKPT)
pairs = [semeval_to_pair(r) for r in test]; y = [r["label"] for r in test]
t0 = time.perf_counter(); preds = det.predict(pairs); dt = time.perf_counter() - t0
from dl.evaluation.common import pred_to_directional
yp = [pred_to_directional(p) for p in preds]
print(classification_report(y, yp, labels=LABELS, digits=4, zero_division=0))
m = metrics(y, yp, LABELS); print("accuracy %.4f | macro-F1 %.4f | weighted-F1 %.4f | %.2f ms/sentence" % (m["accuracy"], m["macro_f1"], m["weighted_f1"], 1000*dt/len(pairs)))
ConfusionMatrixDisplay.from_predictions(y, yp, labels=LABELS, cmap="Blues", xticks_rotation=20); plt.title("Confusion matrix - SemEval gold test"); plt.show()

## 16. Sample predictions

In [ ]:
rows = []
for r, p in list(zip(test, preds))[:400]:
    if r["label"] != "NONE" or p.label.relation_class.value == "causal":
        rows.append({"sentence": r["text"][:110], "gold": r["label"], "pred": pred_to_directional(p), "conf": round(p.label.confidence, 3)})
    if len(rows) >= 12: break
pd.set_option("display.max_colwidth", 120); display(pd.DataFrame(rows))
demo = ["Heavy [E1] smoking [/E1] causes [E2] lung cancer [/E2] in long-term users .",
        "[E2] Insulin resistance [/E2] is driven by chronic [E1] obesity [/E1] .",
        "[E1] Coffee [/E1] was served next to the [E2] cake [/E2] ."]
pr = det.predict_texts(demo)
for t, p in zip(demo, pr): print(f"{t}\n   -> {LABELS[int(p.argmax())]}  {np.round(p.numpy(), 3)}")

## 17. AML vs DL comparison
`dl/evaluation/compare.py` runs spaCy-Rules, the keyword heuristic and the DistilBERT checkpoint on the **identical** 2,700 SemEval gold test sentences (association predictions are projected to `NONE`, since SemEval has no association class), and on the 250 diabetes **silver** pairs (+ the saved 20-pair Gemini/Groq pilot).
**Gold and silver results are different kinds of evidence and are reported separately.**

In [ ]:
from dl.evaluation import compare
compare.main(["--checkpoint", str(CKPT)])
res = json.load(open(ROOT / f"dl/results/{TAG}/comparison_results.json"))
print("model kind:", res["model_kind"])
print("\n=== GOLD (SemEval-2010 Task 8 test, human-annotated) ===")
g = pd.DataFrame({k: {"accuracy": v["accuracy"], "macro-P": v["macro_precision"], "macro-R": v["macro_recall"], "macro-F1": v["macro_f1"], "weighted-F1": v["weighted_f1"],
                      "causal-vs-not F1": v["binary_causal_f1"], **{f"F1 {l}": v["per_class"][l]["f1"] for l in LABELS}, "ms/sent": v["inference_ms_per_sentence"]} for k, v in res["gold_semeval"].items()}).T
display(g.round(4))
print("training seconds (DistilBERT):", res["gold_semeval"]["FineTuned-DistilBERT"].get("training_seconds"))
print("\n=== SILVER (diabetes, rule-generated pseudo labels - NOT human ground truth) ===")
s = pd.DataFrame({k: {"accuracy": v["accuracy"], "macro-F1": v["macro_f1"], "weighted-F1": v["weighted_f1"], "F1 causal": v["per_class"]["causal"]["f1"], "F1 assoc": v["per_class"]["association_only"]["f1"], "F1 none": v["per_class"]["none"]["f1"]}
                  for k, v in res["silver_diabetes"]["all_250_pairs"].items()}).T; display(s.round(4))
print(res["silver_diabetes"]["NOTE"])
print("\n20-pair pilot (where Gemini/Groq predictions exist):"); display(pd.DataFrame({k: {"n": v["n"], "accuracy": v["accuracy"], "macro-F1": v["macro_f1"]} for k, v in res["silver_diabetes"]["pilot_20_pairs"].items()}).T.round(4))
print("NOTE: the saved Gemini/Groq pilot predictions overlap the 250 silver pairs in only n pairs (see column n). That is far too few to compare LLMs with anything; no conclusion should be drawn from this table.")

In [ ]:
from IPython.display import Image, display as d2
for f in ["gold_macro_f1.png", "gold_binary_f1.png", "silver_macro_f1.png", "inference_time.png"]:
    d2(Image(filename=str(ROOT / f"dl/results/{TAG}/{f}"), width=520))

**Reading the comparison (apply to whichever mode printed above).**
* Rule-based extraction is precise when the sentence contains an explicit trigger ("caused by", "leads to"); it needs no training and runs fastest, but misses relations expressed without a listed trigger and cannot learn new phrasing.
* The keyword heuristic ignores syntax and entity roles entirely, so it over-fires on any sentence containing words like *increase/reduce*.
* A fine-tuned *pretrained* transformer is expected to generalise to un-listed phrasings and to use the entity markers for direction, at the cost of training, GPU/MPS compute and ~66M parameters. A *randomly-initialised* model (smoke mode) cannot do this: with ~7k sentences it has no language knowledge to transfer, which is precisely why pretraining matters.
* Silver labels were produced by rules, so spaCy-Rules scores are partly circular on them; they must not be used to rank extractors.

## 18. Graph integration
The DistilBERT predictions are run on the diabetes candidate pairs and fed to the **existing** `EvidenceGraphBuilder`. Policy: confidence threshold; self-loops rejected; duplicate evidence per sentence merged; doc-id/sentence-id/char-spans kept; direction conflicts logged **before** deterministic min-confidence cycle repair; every repair logged; final graph asserted acyclic.

In [ ]:
from dl import run_graph_integration as RG
RG.main(["--checkpoint", str(CKPT), "--threshold", "0.7" if PRETRAINED else "0.4"])
gdir = ROOT / f"dl/results/graph_{TAG}"
log = json.load(open(gdir / "integration_log.json"))
print("direction conflicts logged:", len(log["direction_conflicts"]), "| cycle-repair removals:", len(log["cycle_repair"]))
for r in log["cycle_repair"][:5]: print("  removed", r["removed_edge"], "conf", r["confidence"], "cycle", r["cycle_detected"])
if (gdir / "dl_causal_dag.png").exists(): d2(Image(filename=str(gdir / "dl_causal_dag.png"), width=620))

## 19. Bayesian reasoning demonstration
A DAG from text is a **hypothesis**, not a validated causal model. CPDs here are **illustrative** (noisy-OR style), not learned; the real AML DAG has a single edge. `do()` answers assume the DAG is right and there is no unmeasured confounding.
Demo (B) uses a *hypothetical* DAG with Age confounding BMI→Outcome to show why `P(Y|X)` ≠ `P(Y|do(X))`.

In [ ]:
from dl import run_bayesian_demo as RB
RB.main()
for f in ["bn_real_aml_dag.png", "bn_hypothetical_dag.png"]: d2(Image(filename=str(ROOT / "dl/results/bayesian" / f), width=420))

## 20. Limitations
1. **Domain shift:** training data is general-English SemEval nominals; the target is biomedical abstracts. No human-verified diabetes causal gold exists in the project, so domain performance is *unmeasured*.
2. **Label scheme:** SemEval has no association-only class → the DL model cannot output it; association edges still come from the rules extractor.
3. **Causal ≠ extracted:** a sentence *claiming* causation is not evidence the relation is causal; observational language, hedging and polarity are not modelled (defaults are flagged in `raw_output`).
4. **Silver labels** are rule-generated (4 causal/250); they cannot rank extractors fairly.
5. **Bayesian layer:** illustrative parameters; do-calculus requires correct structure and no hidden confounders.
6. Sentence-level only (no cross-sentence coreference); entity pairs come from the AML candidate generator.
7. This notebook's printed numbers are from the mode shown in the banner; smoke-mode numbers are not DistilBERT results.

## 21. Conclusion
A complete, tested pipeline now exists: leakage-safe gold data → entity-marked DistilBERT with entity pooling → AML-compatible predictions → logged, acyclic evidence graph → pgmpy network with explicit causal caveats. The trustworthy headline result is the **SemEval gold-test comparison produced by running this notebook with pretrained weights on the MacBook**.

## 22. Future scope
Human-annotate a few hundred diabetes sentences (the repo already ships a 50-pair queue) and fine-tune/evaluate in-domain; try biomedical encoders (BioBERT / PubMedBERT); add an association class from BioRED-style data; model polarity and hedging; calibrate confidences; fit CPDs from BRFSS/NHANES with proper adjustment sets.